In [1]:

import numpy as np
import matplotlib.pylab as plt

def get_start_end_is_of_ones_in_binary_array( binary_array):
    '''
    get frames in which consecutive chunk of good midlines or a "midline_chunk" starts and ends
    '''
    binary_array_buffered = np.concatenate([np.zeros(1),binary_array, np.zeros(1)])
    chunk_start_is = np.argwhere(np.diff(binary_array_buffered)==1).flatten()
    chunk_end_is = np.argwhere(np.diff(binary_array_buffered)==-1).flatten()

    return chunk_start_is, chunk_end_is


def multirange(start_indices, end_indices, fsize):
    result = np.zeros(fsize, dtype=int)
    start_indices = start_indices.flatten().astype(np.int32)
    end_indices = end_indices.flatten().astype(np.int32)
    for i in range(start_indices.shape[0]):
        start = start_indices[i]
        end = end_indices[i]
        result[start:end] = 1
    return result

def get_is_long_enough_track(bin_event_track, min_duration_thres, n_frames):
    bin_event_track = bin_event_track.astype('uint8')
    event_start_is, event_end_is = get_start_end_is_of_ones_in_binary_array(bin_event_track)
    event_durations  = event_end_is - event_start_is

    sufficient_length_events_is = np.argwhere(event_durations>min_duration_thres)
    new_event_starts = event_start_is[sufficient_length_events_is]
    new_event_ends = event_end_is[sufficient_length_events_is]
    updated_events_bin_track = multirange(new_event_starts, new_event_ends, n_frames)
    
    too_short_events_is = np.argwhere(event_durations<=min_duration_thres)
    too_short_event_starts = event_start_is[too_short_events_is]
    too_short_event_ends = event_end_is[too_short_events_is]
    short_events_bin_track  = multirange(too_short_event_starts, too_short_event_ends, n_frames)
    
    return short_events_bin_track, updated_events_bin_track, too_short_event_starts, too_short_event_ends
    

def remove_single_frame_behs(all_tracks_feature_mat, feature_to_index, min_event_duration_thres):
    n_frames, n_tracks =  all_tracks_feature_mat[feature_to_index["fwd"]].shape
       
    beh_etho = np.zeros((n_frames, n_tracks))
    beh_etho[:] = np.nan
    beh_etho[all_tracks_feature_mat[feature_to_index["fwd"]] ==1] = feature_to_index["fwd"]
    beh_etho[all_tracks_feature_mat[feature_to_index["rev"]] ==1] = feature_to_index["rev"]
    beh_etho[all_tracks_feature_mat[feature_to_index["turn"]] ==1] = feature_to_index["turn"]
    beh_etho[all_tracks_feature_mat[feature_to_index["pause"]] ==1] = feature_to_index["pause"]
    #pre_post_event_duration_thres = min_event_duration_thres+1
    index_to_feature = {}
    for feature, index in feature_to_index.items():
        index_to_feature[index] = feature
    for event_label in ["pause", "turn", "rev", "fwd"]:
        event_mat =  all_tracks_feature_mat[feature_to_index[event_label]]
        for track_i in range(n_tracks):
            too_short_pause_track, long_enough_pause_track, too_short_event_starts, too_short_event_ends = get_is_long_enough_track(event_mat[:,track_i], min_event_duration_thres-1, n_frames)
            too_short_event_starts = too_short_event_starts.flatten()
            too_short_event_ends = too_short_event_ends.flatten()
            for i in range(too_short_event_starts.shape[0]):
                too_short_event_start = too_short_event_starts[i]
                too_short_event_end = too_short_event_ends[i]
                
                if too_short_event_start>min_event_duration_thres:
                    beh_etho_pre_event = beh_etho[too_short_event_start-min_event_duration_thres:too_short_event_start, track_i]#get beh pre too short rev
                    if np.all(~np.isnan(beh_etho_pre_event)) and np.all(np.diff(beh_etho_pre_event)==0):#if was all the same event prior to too short stim chane event to prior
                        pre_event_label = index_to_feature[beh_etho_pre_event[0]]
                        all_tracks_feature_mat[feature_to_index[pre_event_label]][too_short_event_start:too_short_event_end, track_i] = 1
                        all_tracks_feature_mat[feature_to_index[event_label]][too_short_event_start:too_short_event_end, track_i] = 0
                        beh_etho[too_short_event_start:too_short_event_end, track_i] = beh_etho_pre_event[0]
                        continue
                post_event_transtion = min(too_short_event_end+min_event_duration_thres, n_frames-1)
                beh_etho_post_event = beh_etho[too_short_event_end:post_event_transtion, track_i]
                if np.all(~np.isnan(beh_etho_post_event)) and np.all(np.diff(beh_etho_post_event)==0):#if was all the same event post the too short stim then cahgne to post event
                #if np.all(~np.isnan(beh_etho_post_event)) and np.all(np.diff(beh_etho_post_event)==0):
                    post_event_label = index_to_feature[beh_etho_post_event[0]]
                    all_tracks_feature_mat[feature_to_index[post_event_label]][too_short_event_start:too_short_event_end, track_i] = 1
                    all_tracks_feature_mat[feature_to_index[event_label]][too_short_event_start:too_short_event_end, track_i] = 0
                    beh_etho[too_short_event_start:too_short_event_end, track_i] = beh_etho_post_event[0]
                    continue




def get_is_long_enough_mat(bin_event, min_duration_thres, n_frames):
    n_frames, n_tracks = bin_event.shape
    updated_events_bin = np.zeros( (n_frames, n_tracks))*np.nan
    for track in range(n_tracks):
        bin_event_track = bin_event[:, track]
        event_start_is, event_end_is = get_start_end_is_of_ones_in_binary_array(bin_event_track)
        event_durations  = event_end_is - event_start_is
        
        sufficient_length_events_is = np.argwhere(event_durations>min_duration_thres)
        new_event_starts = event_start_is[sufficient_length_events_is]
        new_event_ends = event_end_is[sufficient_length_events_is]
        updated_events_bin_track = multirange(new_event_starts, new_event_ends, n_frames)
        updated_events_bin[:, track] = updated_events_bin_track

    return None, updated_events_bin
    
def calculate_angles(arr1, arr2):
    magnitude_arr1 = np.linalg.norm(arr1, axis=-1)
    magnitude_arr2 = np.linalg.norm(arr2, axis=-1)

    # Handle zero-length vectors
    mask = (magnitude_arr1 * magnitude_arr2) != 0
    cos_angles = np.empty_like(magnitude_arr1)
    cos_angles[mask] = np.clip(
        np.sum(arr1[mask] * arr2[mask], axis=-1) / (magnitude_arr1[mask] * magnitude_arr2[mask]),
        -1.0, 1.0
    )
    cos_angles[~mask] = np.nan

    angles_radians = np.arccos(cos_angles)
    angles_degrees = np.degrees(angles_radians)
    
    return angles_degrees



def get_is_turning(
       
                   heads, tails, midpoints, 
                   is_looping_track,
                   min_duration_thres = 3, 
                   px_to_um = 15.4
                   ):

    n_frames = heads.shape[0]
    # n_tracks =  heads.shape[1]

    # HT_angles = np.zeros(n_frames, n_tracks)
    HM_vecs = heads-midpoints
    TM_vecs = tails-midpoints
        
    HT_angles = calculate_angles(HM_vecs, TM_vecs)


    is_narrow_angle = abs(HT_angles) < 45
    HM_vec_lengths = np.linalg.norm(HM_vecs, axis =-1)
    TM_vec_lengths = np.linalg.norm(TM_vecs, axis =-1)
    length_buffer = 3*px_to_um
    is_HM_greater_than_TM = HM_vec_lengths<TM_vec_lengths+length_buffer
    
    meets_omega_criteria = np.logical_and(is_narrow_angle,is_HM_greater_than_TM)
    meets_omega_criteria[is_looping_track==1] = 1
    short_events_bin_track, long_enough_omegas = get_is_long_enough_mat(meets_omega_criteria, min_duration_thres, n_frames)


    return long_enough_omegas

        
def calculate_velocity(centroids, dframes, dt):
    velocity_full = np.zeros_like(centroids)*np.nan
    dposition = centroids[dframes:,:]-centroids[:-1*dframes,:] #four frames later minus for frames earlier
    velocity = dposition/dt
    velocity_full[dframes:] = velocity
    return velocity_full

def calculate_head_tail_vector( global_head_coords, global_tail_coords ):

    head_tail_vector = global_head_coords-global_tail_coords
    return head_tail_vector


def dot_product(vec1,vec2, decimals_to_round_to = 3 ):
    dot = np.sum(vec1 * vec2, axis=-1)
    return dot


def calculate_speed(velocity, global_head_coords, global_tail_coords ):
    #get magnitude of speed based on norm of velocity
    speed = np.linalg.norm(velocity, axis = -1)

    #get head tail vector
    head_tail_vector = calculate_head_tail_vector(global_head_coords, global_tail_coords)

    dot = dot_product(velocity,head_tail_vector, decimals_to_round_to = 3 )
    speed[dot<0] = -speed[dot<0]
    return speed


def get_speed(dframes,dt, centroids,
              global_head_coords, global_tail_coords 
              ):
    velocity = calculate_velocity(centroids, dframes, dt)
    speed = calculate_speed(velocity, global_head_coords, global_tail_coords )
    return speed, velocity






def bin_mat_from_start_end_is(mat_shape, start_end_is):
    bin_mat = np.zeros(mat_shape)
    for track, start, end in start_end_is:
        bin_mat[start:end, track ] = 1
    return bin_mat

def get_runs_of_ones(arr, track_is=None):

    runs = []
    n_frames, n_tracks = arr.shape

    if track_is is None:
        track_is = np.arange(n_tracks)

    for track in track_is:

        data = arr[:, track]

        # IMPORTANT
        data = np.nan_to_num(data, nan=0)

        padded = np.pad(data, (1, 1), mode='constant')
        diff = np.diff(padded)

        starts = np.where(diff == 1)[0]
        ends = np.where(diff == -1)[0]

        for start, end in zip(starts, ends):
            runs.append(np.array([track, start, end]))

    return np.stack(runs, axis=1).T



def is_pausing1(speed_mat, turn_mat, speed_thres,
                ):
    
    
    pause_mat = np.zeros(speed_mat.shape)
    pause_mat[np.abs(speed_mat)<speed_thres] = 1
    pause_mat[turn_mat==1] = 0
    
    return  pause_mat


def get_behavior( 
                 centroids,
                 heads, tails, midpoints, 
                 is_looping, 
                 speed_thres = 2.6*23.1,
                 dframes = 4, 
                 fps = 6, 
                 speed = None
                 
                 ):
    dframes = int(dframes)
    
    dt = dframes/fps
    velocity = None
    if speed is None:
        speed, velocity = get_speed(dframes,
                                    dt, 
                                    centroids, 
                                    heads, 
                                    tails
                          )
        
        speed[:dframes] = np.nan

    turning_beh =  get_is_turning(heads, tails, midpoints, 
                  is_looping, min_duration_thres = 3)

    pausing_beh = np.zeros(speed.shape)*np.nan
    pausing_beh = is_pausing1(speed, turning_beh, speed_thres,
              
                )
    
    rev_beh = np.zeros(speed.shape)*np.nan
    rev_beh[speed<0] = 1
    rev_beh[pausing_beh==1] = 0
    rev_beh[turning_beh==1] = 0
        
    fwd_beh = np.zeros(speed.shape)*np.nan
    fwd_beh[speed>0] = 1
    fwd_beh[pausing_beh==1] = 0
    fwd_beh[turning_beh==1] = 0
    

    
    return speed, velocity, fwd_beh, rev_beh, turning_beh,  pausing_beh



def update_fwd_rev_bin(pause_bin, turn_bin, speed_mat):
    turn_pause = np.logical_or(pause_bin==1, turn_bin==1).astype('float')
    rev_mat = np.logical_and(speed_mat<0, np.logical_not(turn_pause)).astype('float')
    fwd_mat = np.logical_and(speed_mat>=0, np.logical_not(turn_pause)).astype('float')
    fwd_mat[np.isnan(speed_mat)] = np.nan
    rev_mat[np.isnan(speed_mat)] = np.nan
    return  fwd_mat, rev_mat


def get_beh_from_bin(fwd, rev, turn, pause):
    label_to_beh = {
        -1: "nan",
        0: "fwds", 
        1:"rev", 
        2:"turn", 
        3: "pause"
    }
    
    behs = np.ones(fwd.shape)*-1
    
    behs[fwd ==1] = 0 
    behs[rev ==1] = 1
    behs[turn==1] = 2
    behs[pause==1] = 3
    return behs, label_to_beh


In [ ]:
import ast
import os

import glob
import matplotlib.pyplot as plt
import pandas as pd


'''load csvs'''
import os 



basename = ""
csv_save_dir = os.path.join(basename, "fetures/")


is_looping_csv = os.path.join(csv_save_dir, "is_looping.csv")

is_looping = np.loadtxt(is_looping_csv, delimiter=",")
n_frames, n_tracks = is_looping.shape

label_to_mat = {"is_looping": is_looping}


for feature_label in ["centroids", 
                      "head_coord", 
                      "tail_coord", 
                      "mdpt_coord"
                      
                      ]:
    feature_csv =  os.path.join(csv_save_dir,f"{feature_label}.csv")
    data = np.loadtxt(feature_csv, delimiter=",")
    
    label_to_mat[feature_label] = data.reshape(n_frames, n_tracks, 2)




#############

speed_thres_um = 2.6*15.4#50

n_frames, n_tracks = is_looping.shape
featureID ="speed"

                 
speed_og,  velocity, fwd, rev, turn, pause =  get_behavior( 
                label_to_mat["centroids"],
                label_to_mat["head_coord"],   
                label_to_mat["tail_coord"],   
                label_to_mat["mdpt_coord"],
                label_to_mat["is_looping"], 
                speed_thres = 2.6*23.1/1.5, 
                dframes = 4, 
                fps = 6, 
                
                )


'''save behs as txt'''
beh, label_to_beh = get_beh_from_bin(fwd, rev, turn, pause)

label_to_beh = {
    0: "fwd", 
    1:"rev", 
    2:"turn", 
    3: "pause"
}
feature_to_index = {
 val:key for key, val in label_to_beh.items()
}

min_event_duration_thres = 3


all_tracks_feature_mat = [(beh.astype(int) == label) for label in [0, 1, 2, 3]]
n_frames = 14880

# # Build orig BEFORE
# beh_etho_orig = np.full((n_frames, n_tracks), np.nan)
# print(all_tracks_feature_mat[0].shape, beh_etho_orig.shape)
# beh_etho_orig[all_tracks_feature_mat[0]] = 0
# beh_etho_orig[all_tracks_feature_mat[1]] = 1
# beh_etho_orig[all_tracks_feature_mat[2]] = 2
# beh_etho_orig[all_tracks_feature_mat[3]] = 3

# print(beh.shape, beh_etho_orig.shape)


# Smooth — use the returned beh_etho directly, don't rebuild it
min_event_duration_thres = 3
remove_single_frame_behs(
    all_tracks_feature_mat, feature_to_index, min_event_duration_thres)


beh_etho_smoothed = np.full((n_frames, n_tracks), np.nan)
print(all_tracks_feature_mat[0].shape, all_tracks_feature_mat[0].shape)
beh_etho_smoothed[all_tracks_feature_mat[0]] = 0
beh_etho_smoothed[all_tracks_feature_mat[1]] = 1
beh_etho_smoothed[all_tracks_feature_mat[2]] = 2
beh_etho_smoothed[all_tracks_feature_mat[3]] = 3



(14880, 37) (14880, 37)


In [3]:


trackIDs = []
# trackIDs_save_dir = csv_save_dir
with open(os.path.join(csv_save_dir, "trackIDs.txt"), "r", encoding="utf-8") as file:
    for line in file:
        trackID_str, frame_start, frame_end = ast.literal_eval(line.strip())
        print(trackID_str, frame_start, frame_end)
 
        trackIDs.append((trackID_str, frame_start, frame_end))

print(trackIDs)

0_13 0 4734
3129_7 3129 5164
3359_8 3359 4584
4786_8 4786 5916
0_2 0 6735
0_12 0 3342
0_3 0 6912
3423_10 3423 10511
0_8 0 2924
0_10 0 12788
0_11 0 687
1096_5 1096 2546
7897_8 7897 13660
865_8 865 5865
3455_5 3455 5017
0_9 0 687
6636_1 6636 6925
0_5 0 3308
2997_7 2997 3357
0_4 0 14848
2997_6 2997 5469
13548_9 13548 14848
6929_2 6929 10455
4700_7 4700 6607
13207_11 13207 14848
6903_3 6903 11643
13952_5 13952 14848
6040_9 6040 7866
7897_7 7897 14848
5872_2 5872 7867
12776_6 12776 14790
865_7 865 2978
7755_5 7755 12728
6929_1 6929 8111
3328_12 3328 7688
0_7 0 3119
2540_6 2540 2892
[('0_13', 0, 4734), ('3129_7', 3129, 5164), ('3359_8', 3359, 4584), ('4786_8', 4786, 5916), ('0_2', 0, 6735), ('0_12', 0, 3342), ('0_3', 0, 6912), ('3423_10', 3423, 10511), ('0_8', 0, 2924), ('0_10', 0, 12788), ('0_11', 0, 687), ('1096_5', 1096, 2546), ('7897_8', 7897, 13660), ('865_8', 865, 5865), ('3455_5', 3455, 5017), ('0_9', 0, 687), ('6636_1', 6636, 6925), ('0_5', 0, 3308), ('2997_7', 2997, 3357), ('0_4', 0

In [4]:


save_dir_name = os.path.join(basename, "outputs/")
os.makedirs(save_dir_name, exist_ok = True)
plate_label = "c3_042125_RIMpReaChR_RIBHisCl_atr0his0"

for track_i, (trackID_str, frame_start, frame_end) in enumerate(trackIDs):

    beh_smoothed = beh_etho_smoothed[frame_start:frame_end, track_i]
    beh_txt = os.path.join(save_dir_name,f"{plate_label}_{trackID_str}_beh.txt")
    np.savetxt(beh_txt, beh_smoothed, delimiter = ',')  

                